# Multilingual Speech Recognition

Whisper Small based multilingual ASR pipeline.

In [ ]:
!pip install -q transformers datasets torchaudio librosa jiwer

In [ ]:
import pandas as pd
import numpy as np
import torch
import librosa
from tqdm import tqdm
from transformers import WhisperProcessor, WhisperForConditionalGeneration

In [ ]:
train_df = pd.read_csv('/kaggle/input/competitions/multilingual-speech-recognition/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/multilingual-speech-recognition/test.csv')
train_df.head(), test_df.head()

In [ ]:
def load_audio(file_path):
    audio, sr = librosa.load(file_path, sr=16000)
    return audio

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
processor = WhisperProcessor.from_pretrained('openai/whisper-small')
model = WhisperForConditionalGeneration.from_pretrained('openai/whisper-small')
model.to(device)
model.eval()

In [ ]:
def transcribe(audio_path):
    audio = load_audio(audio_path)
    inputs = processor(audio, sampling_rate=16000, return_tensors='pt')
    input_features = inputs.input_features.to(device)
    with torch.no_grad():
        predicted_ids = model.generate(input_features)
    return processor.batch_decode(predicted_ids, skip_special_tokens=True)[0].lower()

In [ ]:
predictions = []
for audio_file in tqdm(test_df['audio']):
    path = f'/kaggle/input/competitions/multilingual-speech-recognition/competition_data/train/{audio_file}'
    predictions.append(transcribe(path))
submission = pd.DataFrame({'audio': test_df['audio'], 'text': predictions})
submission.to_csv('submission.csv', index=False)
submission.head()